# Vehicle Length from VS13 Spectrograms

Interactive notebook mirroring `length_estimation/run.py`.

**Prerequisites:** Download VS13 into `length_estimation/data/vs13/{VehicleName}/`.

See `ref_docs/vehicle_length_from_spectrogram.md` for the full plan.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "length_estimation").exists() and (ROOT.parent / "length_estimation").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from length_estimation.config import DEFAULT_OUTPUT_DIR
from length_estimation.src.preprocess import load_clips, resolve_data_dir
from length_estimation.src.features_physics import extract_clip_features
from length_estimation.src.evaluate import run_phase_a
from length_estimation.src.models import run_phase_b_lovo
from length_estimation.config import PhaseBConfig

import pandas as pd
from tqdm import tqdm

OUTPUT_DIR = DEFAULT_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Scans data/vs13 automatically — no separate 'index' step required
records = load_clips(write_manifest=True)
print(f"{len(records)} clips from {resolve_data_dir()}")

## 1. Load dataset (auto-scan — index step not required)

In [ ]:
manifest = pd.DataFrame([{
    "clip_id": r.clip_id, "vehicle": r.vehicle, "speed_kmh": r.speed_kmh,
    "cpa_time_s": r.cpa_time_s, "length_m": r.length_m, "split": r.split,
} for r in records])
manifest.head()

## 2. Phase A — hand-crafted features

In [ ]:
rows = [extract_clip_features(r) for r in tqdm(records)]
features_df = pd.DataFrame(rows)
features_df.to_csv(OUTPUT_DIR / "features.csv", index=False)
features_df.filter(regex="width_x_speed|xcorr_lag|centroid_delta|reassigned_doppler").head()

In [ ]:
results = run_phase_a(features_df, OUTPUT_DIR / "phase_a", target="length_m")
results["correlation"].head(10)

## 3. Phase B — CNN (optional)

Requires PyTorch + GPU recommended. Skip until Phase A shows signal.

In [ ]:
cfg = PhaseBConfig(spec_type="mel", epochs=40, batch_size=16)
folds = run_phase_b_lovo(records, OUTPUT_DIR / "phase_b", cfg, target="length_m", device="cpu")
folds